# Personalisierung des besten Models (RF)

- Baseline Normalisierung
- Kalibrierung auf wenigen Trainingssamples der Zielperson
- Voll personalisiert, nur auf Daten der Zielperson 

## Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scripts.myml import loso_binary_baseline_check_nested_cv
from sklearn.ensemble import RandomForestClassifier

## Daten laden

In [2]:
DATASET_PATH = Path("dataset/np-dataset")
X = np.load(DATASET_PATH / "X.npy")
X_catch22 = np.load(DATASET_PATH / "X_catch22.npy")
X_catch22_feature_names = np.load(DATASET_PATH / "feature_names.npy")
y_heater = np.load(DATASET_PATH / "y_heater.npy")
subjects = np.load(DATASET_PATH / "subjects.npy")
y = np.argmax(y_heater, axis=1)

# EMG sensor bei 42 war defekt
valid = subjects != 42
X_catch22 = X_catch22[valid]
y = y[valid]
subjects = subjects[valid]

In [3]:
print(f"X shape : {X_catch22.shape}")
print(f"y shape : {y.shape}   classes: {np.unique(y).tolist()}")
print(f"subjects : {np.unique(subjects).shape[0]} unique subjects")
print(f"Class counts : { {c: int((y==c).sum()) for c in range(6)} }")

X shape : (2447, 174)
y shape : (2447,)   classes: [0, 1, 2, 3, 4, 5]
subjects : 51 unique subjects
Class counts : {0: 408, 1: 408, 2: 408, 3: 407, 4: 408, 5: 408}


## Hyperparameter




In [4]:
# Random Forest
PARAM_GRID_RF = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
}

INNER_N_SPLITS = 5
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Feature-Matrixes

In [5]:
# without normalization
X_catch22 = X_catch22.copy()

print(np.nanmean(X_catch22))

40.55045877855115


## Setup for binary classification

In [6]:
class_comparison = (0, 5)
mask = (y==class_comparison[0]) | (y==class_comparison[1])
X_filtered = X_catch22[mask]
y_filtered = y[mask]
group_labels = subjects[mask]
y_binary = (y_filtered == class_comparison[1]).astype(int)

## Ohne Data Leakage Test (RF)

- baseline normalisierung ohne leakeage -> k=3 samples von 0 class werden genommen und auf diesen wird die baseline_normalisierung ausgeführt. Auf den restslich 5 + 8-class5 samples wird getestet

In [7]:
rf = RandomForestClassifier(random_state=RANDOM_STATE)
rf_baseline_norm_without_leakage = loso_binary_baseline_check_nested_cv(
    X_filtered,
    y_binary,
    group_labels,
    rf,
    space=PARAM_GRID_RF,
    k_baseline=3,
    model_type="classifier"
)

print(f"    Accuracy : {rf_baseline_norm_without_leakage['accuracy']:.3f} ± {rf_baseline_norm_without_leakage['accuracy_std']:.3f}")
print(f"    AUC      : {rf_baseline_norm_without_leakage['auc']:.3f} ± {rf_baseline_norm_without_leakage['auc_std']:.3f}")
print(f"    F1       : {rf_baseline_norm_without_leakage['f1']:.3f} ± {rf_baseline_norm_without_leakage['f1_std']:.3f}")
print(f"    Sensitivity : {rf_baseline_norm_without_leakage['sensitivity']:.3f} ± {rf_baseline_norm_without_leakage['sensitivity_std']:.3f}")
print(f"    Specificity : {rf_baseline_norm_without_leakage['specificity']:.3f} ± {rf_baseline_norm_without_leakage['specificity_std']:.3f}")

    Accuracy : 0.943 ± 0.069
    AUC      : 0.978 ± 0.051
    F1       : 0.951 ± 0.061
    Sensitivity : 0.938 ± 0.095
    Specificity : 0.949 ± 0.104
